# 1. Environment Setup & Data Downloading
Install dependencies, configure Kaggle API, and download the HAM10000 dataset.


In [15]:
# Install dependencies
!pip install -q kaggle scikit-learn matplotlib seaborn keras pandas numpy Pillow

# ── Paste your Kaggle credentials here ───────────────────
KAGGLE_USERNAME = "ethanharter"
KAGGLE_KEY      = "KGAT_ceb37d023f07acfacca73288ff37791f"

import os, json
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
creds_path = os.path.join(kaggle_dir, 'kaggle.json')
with open(creds_path, 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(creds_path, 0o600)

# Download HAM10000
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 \
    --unzip -p /content/ham10000 -q
print('✅ Environment Setup Complete and Dataset downloaded')


Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
✅ Environment Setup Complete and Dataset downloaded


# 2. Imports
Importing required libraries for data processing and model building.


In [16]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import glob
from PIL import Image
np.random.seed(123)

from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D
from keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ReduceLROnPlateau


# 3. Load & Preprocess Data
Map image files to the dataset and perform basic cleaning.


In [17]:
base_skin_dir = '/content/ham10000'

# Create dictionary for image paths
imageid_path_dict = {os.path.splitext(os.path.basename(x))[0]: x
                     for x in glob.glob(os.path.join(base_skin_dir, '**', '*.jpg'), recursive=True)}

# Define human-friendly labels
lesion_type_dict = {
    'nv': 'Melanocytic nevi',
    'mel': 'Melanoma',
    'bkl': 'Benign keratosis-like lesions ',
    'bcc': 'Basal cell carcinoma',
    'akiec': 'Actinic keratoses',
    'vasc': 'Vascular lesions',
    'df': 'Dermatofibroma'
}

skin_df = pd.read_csv(os.path.join(base_skin_dir, 'HAM10000_metadata.csv'))

# Create new columns for readability
skin_df['path'] = skin_df['image_id'].map(imageid_path_dict.get)
skin_df['cell_type'] = skin_df['dx'].map(lesion_type_dict.get) 
skin_df['cell_type_idx'] = pd.Categorical(skin_df['cell_type']).codes

# Clean data: Fill missing ages with the mean
skin_df['age'].fillna((skin_df['age'].mean()), inplace=True)


/tmp/ipykernel_75424/3740135304.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  skin_df['age'].fillna((skin_df['age'].mean()), inplace=True)


# 4. Train-Test Split & Data Generators
Use `ImageDataGenerator.flow_from_dataframe` to read images from disk on-the-fly, avoiding RAM crashes.


In [23]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Drop rows with missing paths just in case
skin_df = skin_df.dropna(subset=['path'])

# Create splits
train_df, test_df = train_test_split(skin_df, test_size=0.20, random_state=1234, stratify=skin_df['cell_type_idx'])
train_df, val_df = train_test_split(train_df, test_size=0.10, random_state=2, stratify=train_df['cell_type_idx'])

# Build Dataset
def make_dataset(dataframe, augment=False):
    paths = dataframe['path'].values
    labels = tf.keras.utils.to_categorical(dataframe['cell_type_idx'].values, num_classes=7)
    
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    
    def process_image(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [75, 100]) # Height, Width
        # Standard [0, 1] scaling
        img = img / 255.0
        return img, label
        
    def augment_fn(img, label):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, max_delta=0.1)
        return img, label
        
    ds = ds.map(process_image, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.shuffle(1000)
    
    ds = ds.batch(10).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, augment=True)
val_ds = make_dataset(val_df, augment=False)
test_ds = make_dataset(test_df, augment=False)


# 5. Define the CNN Architecture
Build a standard CNN using Keras Sequential API.


In [24]:
input_shape = (75, 100, 3)
num_classes = 7

model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', padding='Same', input_shape=input_shape))
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', padding='Same'))
model.add(MaxPool2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(64, (3, 3), activation='relu', padding='Same'))
model.add(Conv2D(64, (3, 3), activation='relu', padding='Same'))
model.add(MaxPool2D(pool_size=(2, 2)))
model.add(Dropout(0.40))

model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 75, 100, 32)    │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 75, 100, 32)    │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 37, 50, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 37, 50, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 37, 50, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 37, 50, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 18, 25, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 18, 25, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 28800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │     3,686,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,752,999 (14.32 MB)

 Trainable params: 3,752,999 (14.32 MB)

 Non-trainable params: 0 (0.00 B)

# 6. Training Configuration
Set up the Adam optimizer and callbacks.


In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau

optimizer = Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-7, amsgrad=False)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

learning_rate_reduction = ReduceLROnPlateau(
    monitor='val_accuracy', 
    patience=3, 
    verbose=1, 
    factor=0.5, 
    min_lr=0.00001
)


# 7. Model Training & Evaluation
Train the model and evaluate.


In [26]:
epochs = 50
history = model.fit(
    train_ds,
    epochs=epochs,
    validation_data=val_ds,
    verbose=1,
    callbacks=[learning_rate_reduction]
)

loss_v, accuracy_v = model.evaluate(val_ds, verbose=1)
loss, accuracy = model.evaluate(test_ds, verbose=1)

print("Validation: accuracy = %f  ;  loss_v = %f" % (accuracy_v, loss_v))
print("Test: accuracy = %f  ;  loss = %f" % (accuracy, loss))

# Save model in Keras format
model.save("/content/dermascan.keras")
print("✅ Model saved to /content/dermascan.keras")

Epoch 1/50


ValueError: None values not supported.

In [ ]:
# # Mount Google Drive and save model there
# from google.colab import drive
# drive.mount('/content/drive')

# import shutil
# shutil.copy('/content/dermascan.keras', '/content/drive/MyDrive/dermascan.keras')
# print('✅ Model saved to Google Drive → My Drive/dermascan.keras')